In [2]:
suppressMessages(library(dplyr))
suppressMessages(library(tidyr))

In [ ]:
dir.create('inputs')

In [ ]:
### get PVALUE sig CREs
celltype = c('Beta','Acinar_4','Acinar_4','Activated_Stellate','Alpha','Endothelial','Quiescent_Stellate','Tcells','Acinar_5','Ductal')
disease_use <- c('earlyT1D','Aab','lateT1D')

## down reg
#dir.create('inputs/upreg')
for (CELL in celltype){
        for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/071824_recalculateLogFC_DiseaseDiff/%s_%s.LogRegOut_recalcFC.txt', CELL,disease_use[disease_num]))
        if (file.exists(file_name)) {
           # print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])

            res <- subset(res, subset = res[pval_col] < 0.05)
            res <- subset(res, subset = L2FC > 0)
            if(nrow(res) == 0)next
            res <- subset(res, select=-c(std.error,estimate))
            res <- select(res, -contains(c("pvalue_")))
            res$celltype <- CELL
            res <- separate(res, peak, into = c("chr", "start", "end"), sep = "_")
            write.table(res, sprintf('inputs/upreg/%s_%s_pval05_upreg.bed',CELL,disease_use[disease_num]), sep = '\t', quote = FALSE, col.names = FALSE, row.names = FALSE)
        }  
    }
} 

## down reg
celltype = c('Beta','Acinar_4','Acinar_4','Activated_Stellate','Alpha','Endothelial','Quiescent_Stellate','Tcells','Acinar_5','Ductal')

dir.create('inputs/downreg')
for (CELL in celltype){
        for (disease_num in 1:length(disease_use)){
        file_name <- paste0(sprintf('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/marker_CREs/012924_markerCREs_unionPeaks/CorrectedUnionPeaks_052724/071824_recalculateLogFC_DiseaseDiff/%s_%s.LogRegOut_recalcFC.txt', CELL,disease_use[disease_num]))
        if (file.exists(file_name)) {
           # print(file_name)
            res <- read.table(file_name, sep = '\t', header = TRUE)
            pval_col <- paste0('pvalue_',disease_use[disease_num])

            res <- subset(res, subset = res[pval_col] < 0.05)
            res <- subset(res, subset = L2FC < 0)
            if(nrow(res) == 0)next
            res <- subset(res, select=-c(std.error,estimate))
            res <- select(res, -contains(c("pvalue_")))
            res$celltype <- CELL
            res <- separate(res, peak, into = c("chr", "start", "end"), sep = "_")
            write.table(res, sprintf('inputs/downreg/%s_%s_pval05_downreg.bed',CELL,disease_use[disease_num]), sep = '\t', quote = FALSE, col.names = FALSE, row.names = FALSE)
        }  
    }
} 


In [ ]:
### get background peaks
dir.create('backgrounds/')
celltype = c('Beta','Acinar_4','Acinar_4','Activated_Stellate','Alpha','Endothelial','Quiescent_Stellate','Tcells','Acinar_5','Ductal')

for (CELL in celltype){
    file_name <- paste0(sprintf('/nfs/lab/projects/nPOD/downstream_files/ATAC/final_peakcall/peakCallOutput_qvalue05_UNparallelized_20230118/%s_peaks.bed',CELL))
    if (file.exists(file_name)) {
            print(file_name)
            file.copy(file_name, 'backgrounds/')
        }  
    }


#### Script to run HOMER
```bash
#/usr/bin/bash

mapfile -t cells < /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/celltypes.txt

N=10
for cell in  ${cells[@]}; do
        ((i=i%N)); ((i++==0)) && wait

        (### up regulated
    for COND in Aab earlyT1D lateT1D; do date; echo "Working on" ${cell}; echo "Working on" ${COND}; findMotifsGenome.pl /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/inputs/upreg/${cell}_${COND}_pval05_upreg.bed hg38 /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/outputs/${cell}_${COND}_pval_upreg/ -size 200 -bg /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/backgrounds/${cell}_peaks.bed -mknown /nfs/lab/welison/References/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer; date; echo "Done"; done
    echo '  DONE WITH PVALUE UPREG'


    ### down regulated
   for COND in Aab earlyT1D lateT1D; do date; echo "Working on" ${cell}; echo "Working on" ${COND}; findMotifsGenome.pl /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/inputs/downreg/${cell}_${COND}_pval05_downreg.bed hg38 /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/outputs/${cell}_${COND}_pval_downreg/ -size 200 -bg /nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/JASPERmotifs_correctLogFCcalc/backgrounds/${cell}_peaks.bed -mknown /nfs/lab/welison/References/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer; date; echo "Done"; done
    echo '  DONE WITH PVALUE DOWNREG'

) done
```

# reformat HOMER outputs to be easier to understand

In [3]:
### code modified from Emily's

In [ ]:
cells<-scan('/nfs/lab/rlmelton/npod/notebooks/sherlock/Downstream_analysis_nPOD_april2022/Publication/Final_Downstream/HOMER/rerun_directionity/celltypes.txt', sep="\n", what="")
direction <- c('upreg','downreg')
disease <- c('Aab','earlyT1D','lateT1D')

In [4]:
#Read in cell types, header, and results (for some reason it doesn't like it when you read the header and table in 
# together.)
results<-list()
for (c in cells){
    for(d in disease){
        for(dir in direction){
                  if(file.exists(sprintf('outputs/%s_%s_pval_%s/knownResults.txt',c, d, dir))){
                    print(sprintf('outputs/%s_%s_pval_%s/knownResults.txt',c, d, dir))
                    df<-suppressMessages(vroom(sprintf('outputs/%s_%s_pval_%s/knownResults.txt',c, d, dir)))
                    colnames(df)<-gsub(" ", "_", colnames(df))
                    df$comparison <- paste0(c,'_',d,'_',dir)
                    df$comparison <- gsub('Acinar_4','Acinar4',df$comparison)
                    df$comparison <- gsub('Acinar_5','Acinar5',df$comparison)
                      
                    df$comparison <- gsub('Activated_Stellate','ActivatedStellate',df$comparison)
                    df$comparison <- gsub('Quiescent_Stellate','QuiescentStellate',df$comparison)
                    results[[paste0(c,'_',d,'_',dir)]]<-df   
            
                    }
        }

    }
   
}
head(results[[1]])
results2 <- results

ERROR: Error in eval(expr, envir, enclos): object 'cells' not found


In [ ]:
#Grabbing just the columns I care about, then creating L2FC and Log10(padj) and filtering for padj < 0.1
sub_res<-lapply(1:length(results), function(i){
    df<-results2[[i]]
    #Drop columns i dont care about
    df2<-df[,c(1,3, 5, 7, 9,10)]
    #prep column values to be numeric so we can calculate fold change
    df2$`%_of_Target_Sequences_with_Motif`<-gsub("[%]", "", df2$`%_of_Target_Sequences_with_Motif`)
    df2$`%_of_Background_Sequences_with_Motif`<-gsub("[%]", "", df2$`%_of_Background_Sequences_with_Motif`)
    df2$`%_of_Target_Sequences_with_Motif`<-as.numeric(df2$`%_of_Target_Sequences_with_Motif`)
    df2$`%_of_Background_Sequences_with_Motif`<-as.numeric(df2$`%_of_Background_Sequences_with_Motif`)
    #Fold change and Log2 Fold change calculation
    FC<-(df2$`%_of_Target_Sequences_with_Motif`)/df2$`%_of_Background_Sequences_with_Motif`
    df2$FC<-FC
    L2FC<-log2(abs(FC))
    df2$L2FC<-L2FC
    #Replacing q-value with our padj (same method but doesn't round to 0 and 1 so we don't get infinity when we 
    # do Log2FC)
    df2$`-Log10p`<- -log(x = df2$`P-value`, base = 10)
    df2$padj<-p.adjust(p = df2$`P-value`, method = "BH", n = nrow(df2)) 
    df2$`-Log10padj`<- -log(x = df2$padj, base = 10)
    #df3<-df2[df2$padj < .1, ]
    df3<-df2[,-3]
    #renaming and rearranging columns because i can't stand the way it looks
    colnames(df3)<-c('Motif_Name','P-value','Perc_Target_Motif','Perc_Background_Motif','comparison','FC','L2FC','-Log10p','padj','-Log10padj')
    df3<-relocate(df3, Motif_Name, FC, L2FC, 'P-value', '-Log10p', padj, '-Log10padj','Perc_Target_Motif','Perc_Background_Motif','comparison')
    return(df3)
})
head(sub_res[[2]])

In [ ]:
#Save
#dir.create('outputs/formatted_results/')
for (i in 1:length(results)){
    df<-sub_res[[i]]
    if(nrow(df) > 1){
       file <- unique(df$comparison)
        #ct<-cells[[i]]
        write.table(df, sprintf('outputs/formatted_results/%s_allResults_homer.tsv',file), sep="\t", quote=FALSE, row.names=FALSE)
  
    }
}

#### nicely format results for excel table

In [129]:
library(stringr)


In [130]:
sub_res2<-lapply(1:length(sub_res), function(i){
    df<-sub_res[[i]]
    #Drop columns i dont care about
    df2<-data.frame(df[,c(1,4,6,8,9,10)])
    x <- strsplit(df2$comparison, split = "_")
    df2$`Cell type` <- x[[1]][1]
    df2$State <- paste0(x[[1]][3],',',x[[1]][2])
    df2$State <- gsub('earlyT1D','T1D_early',df2$State)
    df2$State <- gsub('downreg','Down',df2$State)
    df3<-data.frame(df2[,c(1,2,3,4,5,6,7,8)])

   
    return(df3)
})
head(sub_res2[[2]])
head(sub_res2[[13]])

,Motif_Name,P.value,padj,Perc_Target_Motif,Perc_Background_Motif,comparison,Cell.type,State
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,MA0773.1_MEF2D,0.001,0.2306667,3.63,2.06,Beta_Aab_downreg,Beta,"Down,Aab"
2,MA0799.2_RFX4,0.001,0.2306667,2.30,1.12,Beta_Aab_downreg,Beta,"Down,Aab"
3,MA0660.1_MEF2B,0.001,0.2306667,3.41,2.05,Beta_Aab_downreg,Beta,"Down,Aab"
4,MA0600.2_RFX2,0.010,0.4325000,3.19,2.00,Beta_Aab_downreg,Beta,"Down,Aab"
5,MA1554.1_RFX7,0.010,0.4325000,7.19,5.35,Beta_Aab_downreg,Beta,"Down,Aab"
6,MA0052.4_MEF2A,0.010,0.4325000,11.19,8.91,Beta_Aab_downreg,Beta,"Down,Aab"


,Motif_Name,P.value,padj,Perc_Target_Motif,Perc_Background_Motif,comparison,Cell.type,State
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
1,MA1128.1_FOSL1::JUN,0.001,0.04942857,10.00,6.87,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"
2,MA1137.1_FOSL1::JUNB,0.001,0.04942857,9.20,6.21,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"
3,MA1634.1_BATF,0.001,0.04942857,9.66,6.63,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"
4,MA0099.3_FOS::JUN,0.001,0.04942857,9.55,6.54,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"
5,MA0462.2_BATF::JUN,0.001,0.04942857,9.66,6.64,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"
6,MA0491.2_JUND,0.001,0.04942857,8.07,5.34,Acinar4_earlyT1D_upreg,Acinar4,"upreg,T1D_early"


In [131]:
#Save
#dir.create('outputs/formatted_results/')
for (i in 1:length(sub_res2)){
    df<-sub_res2[[i]]
    if(nrow(df) > 1){
       file <- unique(df$comparison)
        df3<-data.frame(df[,c(1,2,3,4,5,7,8)])

        #ct<-cells[[i]]
        write.table(df3, sprintf('outputs/formatted_results/%s_formattedResults_homer.tsv',file), sep="\t", quote=FALSE, row.names=FALSE)
  
    }
}